# 01 EDA: Facebook Campaign Performance

This notebook focuses on understanding the Facebook campaign dataset before integrating any external Google Trends signal.

Goals:
- validate the cleaned dataset
- understand KPI distributions and anomalies
- profile age and gender performance
- evaluate audience-cluster coverage and stability
- identify the strongest and weakest audience clusters

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("ggplot")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

In [ ]:
def find_project_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "data" / "processed" / "campaign_data_cleaned.csv").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root from the current notebook working directory.")


BASE_DIR = find_project_root()
CAMPAIGN_PATH = BASE_DIR / "data" / "processed" / "campaign_data_cleaned.csv"
CLUSTER_SUMMARY_PATH = BASE_DIR / "data" / "processed" / "audience_cluster_summary.csv"
BEST_CLUSTER_PATH = BASE_DIR / "data" / "processed" / "audience_cluster_best.csv"
WORST_CLUSTER_PATH = BASE_DIR / "data" / "processed" / "audience_cluster_worst.csv"

campaign_df = pd.read_csv(CAMPAIGN_PATH, parse_dates=["reporting_start", "reporting_end", "date"])
cluster_summary_df = pd.read_csv(CLUSTER_SUMMARY_PATH, parse_dates=["date_first", "date_last"])
best_clusters_df = pd.read_csv(BEST_CLUSTER_PATH)
worst_clusters_df = pd.read_csv(WORST_CLUSTER_PATH)

print("Project data loaded successfully.")
print(f"Campaign rows: {campaign_df.shape[0]:,}, columns: {campaign_df.shape[1]}")
print(f"Cluster summary rows: {cluster_summary_df.shape[0]:,}, columns: {cluster_summary_df.shape[1]}")

## Dataset Overview

In [ ]:
overview = pd.DataFrame(
    {
        "dtype": campaign_df.dtypes.astype(str),
        "missing_values": campaign_df.isna().sum(),
        "missing_pct": campaign_df.isna().mean().mul(100),
    }
).sort_values(["missing_values", "dtype"], ascending=[False, True])

overview

In [ ]:
quality_summary = {
    "date_min": campaign_df["date"].min(),
    "date_max": campaign_df["date"].max(),
    "unique_campaigns": campaign_df["campaign_id"].nunique(dropna=True),
    "unique_audience_clusters": campaign_df["audience_cluster_key"].nunique(),
    "duplicate_rows": int(campaign_df.duplicated().sum()),
    "click_anomalies": int(campaign_df["click_anomaly"].sum()),
    "conversion_anomalies": int(campaign_df["conversion_anomaly"].sum()),
}

pd.Series(quality_summary)

## KPI Distribution Checks

These summary stats help us understand whether the dataset is dominated by outliers or unstable ratios.

In [ ]:
kpi_columns = ["ctr", "conversion_rate", "cpc", "cpa", "roas", "spent", "clicks", "approved_conversion"]
campaign_df[kpi_columns].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).T

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

campaign_df["ctr"].plot(kind="hist", bins=30, ax=axes[0, 0], title="CTR Distribution")
campaign_df["conversion_rate"].plot(kind="hist", bins=30, ax=axes[0, 1], title="Conversion Rate Distribution")
campaign_df["cpc"].dropna().plot(kind="hist", bins=30, ax=axes[1, 0], title="CPC Distribution")
campaign_df["roas"].dropna().plot(kind="hist", bins=30, ax=axes[1, 1], title="ROAS Distribution")

plt.tight_layout()

## Time-Series Health Check

Before joining any external trend data, we need to confirm that the campaign time series is usable and understand the day-level activity pattern.

In [ ]:
daily_df = (
    campaign_df.groupby("date", as_index=False)
    .agg(
        ads=("ad_id", "count"),
        impressions=("impressions", "sum"),
        clicks=("clicks", "sum"),
        spent=("spent", "sum"),
        approved_conversion=("approved_conversion", "sum"),
        revenue=("revenue", "sum"),
    )
)

daily_df["ctr"] = np.divide(
    daily_df["clicks"],
    daily_df["impressions"],
    out=np.zeros(len(daily_df), dtype=float),
    where=daily_df["impressions"] != 0,
)
daily_df["conversion_rate"] = np.divide(
    daily_df["approved_conversion"],
    daily_df["clicks"],
    out=np.zeros(len(daily_df), dtype=float),
    where=daily_df["clicks"] != 0,
)
daily_df["roas"] = np.divide(
    daily_df["revenue"],
    daily_df["spent"],
    out=np.full(len(daily_df), np.nan, dtype=float),
    where=daily_df["spent"] != 0,
)

daily_df

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

axes[0].plot(daily_df["date"], daily_df["spent"], marker="o")
axes[0].set_title("Daily Spend")

axes[1].plot(daily_df["date"], daily_df["approved_conversion"], marker="o", color="tab:green")
axes[1].set_title("Daily Approved Conversions")

axes[2].plot(daily_df["date"], daily_df["roas"], marker="o", color="tab:orange")
axes[2].set_title("Daily ROAS")
axes[2].set_xlabel("Date")

plt.tight_layout()

## Age and Gender Performance

In [ ]:
age_gender_summary = (
    campaign_df.groupby(["age", "gender"], as_index=False)
    .agg(
        ads=("ad_id", "count"),
        impressions=("impressions", "sum"),
        clicks=("clicks", "sum"),
        spent=("spent", "sum"),
        approved_conversion=("approved_conversion", "sum"),
        revenue=("revenue", "sum"),
    )
)

age_gender_summary["ctr"] = np.divide(
    age_gender_summary["clicks"],
    age_gender_summary["impressions"],
    out=np.zeros(len(age_gender_summary), dtype=float),
    where=age_gender_summary["impressions"] != 0,
)
age_gender_summary["conversion_rate"] = np.divide(
    age_gender_summary["approved_conversion"],
    age_gender_summary["clicks"],
    out=np.zeros(len(age_gender_summary), dtype=float),
    where=age_gender_summary["clicks"] != 0,
)
age_gender_summary["roas"] = np.divide(
    age_gender_summary["revenue"],
    age_gender_summary["spent"],
    out=np.full(len(age_gender_summary), np.nan, dtype=float),
    where=age_gender_summary["spent"] != 0,
)

age_gender_summary.sort_values("roas", ascending=False)

In [ ]:
pivot_roas = age_gender_summary.pivot(index="age", columns="gender", values="roas")
pivot_ctr = age_gender_summary.pivot(index="age", columns="gender", values="ctr")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pivot_roas.plot(kind="bar", ax=axes[0], title="ROAS by Age and Gender")
axes[0].set_ylabel("ROAS")

pivot_ctr.plot(kind="bar", ax=axes[1], title="CTR by Age and Gender")
axes[1].set_ylabel("CTR")

plt.tight_layout()

## Audience Cluster Coverage and Stability

Most clusters are extremely sparse, so we need to quantify how concentrated the data is before turning cluster-level rankings into business claims.

In [ ]:
cluster_frequency_distribution = cluster_summary_df["ads"].value_counts().sort_index().rename_axis("ads_per_cluster").reset_index(name="cluster_count")
cluster_frequency_distribution

In [ ]:
coverage_summary = pd.DataFrame(
    {
        "threshold": [1, 2, 3, 5],
        "clusters": [int((cluster_summary_df["ads"] >= threshold).sum()) for threshold in [1, 2, 3, 5]],
        "spend_covered_pct": [
            cluster_summary_df.loc[cluster_summary_df["ads"] >= threshold, "spent"].sum() / cluster_summary_df["spent"].sum() * 100
            for threshold in [1, 2, 3, 5]
        ],
        "approved_conversion_covered_pct": [
            cluster_summary_df.loc[cluster_summary_df["ads"] >= threshold, "approved_conversion"].sum() / cluster_summary_df["approved_conversion"].sum() * 100
            for threshold in [1, 2, 3, 5]
        ],
    }
)

coverage_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cluster_summary_df["ads"].plot(kind="hist", bins=8, ax=axes[0], title="Cluster Frequency Distribution")
axes[0].set_xlabel("Ads per Cluster")

cluster_summary_df["spent"].clip(upper=cluster_summary_df["spent"].quantile(0.95)).plot(
    kind="hist",
    bins=30,
    ax=axes[1],
    title="Cluster Spend Distribution (Trimmed at 95th Percentile)",
)
axes[1].set_xlabel("Spend")

plt.tight_layout()


## Best and Worst Eligible Clusters

These tables use the ranking rule already baked into the summary script: at least 3 ads and total spend >= 10.

In [ ]:
display_columns = [
    "audience_cluster_key",
    "audience_cluster_label",
    "ads",
    "spent",
    "approved_conversion",
    "revenue",
    "ctr",
    "conversion_rate",
    "cpc",
    "cpa",
    "roas",
]

best_clusters_df[display_columns]

In [ ]:
worst_clusters_df[display_columns]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

best_clusters_df.sort_values("roas").plot(
    kind="barh",
    x="audience_cluster_label",
    y="roas",
    ax=axes[0],
    color="tab:green",
    title="Top Eligible Clusters by ROAS",
)

worst_clusters_df.sort_values("roas", ascending=False).plot(
    kind="barh",
    x="audience_cluster_label",
    y="roas",
    ax=axes[1],
    color="tab:red",
    title="Bottom Eligible Clusters by ROAS",
)

plt.tight_layout()

Some bottom-ranked clusters have zero approved conversions. In those cases, revenue is zero and ROAS is exactly zero by construction, so the lowest part of the ranking mixes zero-conversion clusters with low-but-nonzero-ROAS clusters.

## EDA Findings

### 1. Are conversions concentrated in a small number of clusters?
Conversions are somewhat concentrated, but not dominated by only a handful of audience clusters. The top 5 clusters account for 8.8% of approved conversions, the top 10 account for 14.7%, and the top 50 account for 37.3%. At the same time, 38.8% of all clusters generated zero approved conversions. This suggests a long-tail structure: performance is spread across many clusters, but a large share of clusters still underperform or fail to convert at all.

### 2. Do best-performing clusters also show strong CTR, or mainly strong post-click conversion?
The strongest clusters do not win only because they generate more clicks. Their main advantage comes from stronger post-click efficiency. The median CTR among top-performing clusters is only modestly higher than the overall eligible-cluster median, but their median conversion rate is substantially higher. This suggests that the best audience clusters are more effective at turning clicks into approved conversions, not just at attracting attention.

### 3. Are age and gender differences large enough to justify segmentation in the next phase?
Yes. The differences are meaningful enough to justify segmentation in the next phase of the project. Female audiences show a higher CTR overall, which suggests stronger click engagement, while male audiences show a notably higher conversion rate and stronger ROAS. The age-by-gender breakdown also shows material variation, with some groups performing much more efficiently than others. This supports keeping age and gender as important analytical dimensions in future modeling and reporting.

### 4. Does the time series look stable enough to align with an external demand signal?
The time series is stable enough for exploratory alignment with an external demand signal such as Google Trends, but it is not long enough for strong inference. The dataset covers 14 consecutive days with no gaps, and daily spend is relatively stable. ROAS fluctuates more than spend, but the series is still usable for directional analysis. Because the window is short, any relationship with external demand should be framed as exploratory correlation rather than causal evidence.

